# 0. Setup

In [2]:
import ibis
from ibis import _
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_1a_run_panel import run_panel, ModelSpec, format_str
from f_1b_panel_helpers import add_fe

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

# First difference the panel data

In [2]:
%%script fd sample
g_name = 'working_yearly_g'
n_name = 'working_yearly_n'
t_panel_g = con.table(g_name)
t_panel_n = con.table(n_name)

#--- #
t_panel = add_fe(t_panel_g, fe=['t', 'i', 'c'])
t_panel_sample = (
    t_panel
    .select([c for c in t_panel.columns if c.startswith('year_')])
    .order_by(ibis.random())
    .limit(5)
)
display(t_panel_sample.execute())

Couldn't find program: 'fd'


# 1a. LMM
- 26 seconds run nowadays estimating 2 models

In [6]:
def_panel_name = "working_yearly_g"

struct_map = {
    'k': 'beta_1',
    'l': 'beta_2',
    'wg1_y': 'delta_1',
    'wg2_y': 'delta_2',
    'wg3_y': 'delta_3'
}
struct_calc = ['xi_1', 'zeta_1', 'xi_2', 'zeta_2', 'xi_3', 'zeta_3']

models = {
    'exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i', 'c'],
        'category': 'basic',
        'include': False
    },
    'iv': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l'
            ],
        },
        'category': 'basic',
        'include': False
    },
    'iv4': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l',
                'w4g1_k', 'w4g1_l'
            ],
        },
        'category': 'basic'
    },
    
    # OLS 1st-stage tests
    'lag2': {
        'Y': 'wg1_y',
        'X': ['w2g1_k', 'w2g1_l'],
        'fe': ['i', 't'],
        'category': 'ols'
    },
    'lag3': {
        'Y': 'wg1_y',
        'X': ['w2g1_k', 'w2g1_l', 'w3g1_k', 'w3g1_l'],
        'fe': ['i', 't'],
        'category': 'ols'
    },
    'lag4': {
        'Y': 'wg1_y',
        'X': ['w2g1_k', 'w2g1_l', 'w3g1_k', 'w3g1_l', 'w4g1_k', 'w4g1_l'],
        'fe': ['i', 't'],
        'category': 'ols'
    },
    'exog-2': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'W': ['wg2_y', 'wg2_k', 'wg2_l'],
        'fe': ['t', 'i', 'c'],
        'category': '3r',
        'struct_calc': ['xi_1', 'zeta_1', 'xi_2', 'zeta_2'],
        'include': True
    },
    'exog-3': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'W': ['wg2_y', 'wg2_k', 'wg2_l', 'wg3_l', 'wg3_k', 'wg3_y'],
        'fe': ['t', 'i', 'c'],
        'category': '3r',
        'include': False
    },
    'iv-2': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l'
        ],
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l']
        },
        'category': '3r',
        'struct_calc': ['xi_1', 'zeta_1', 'xi_2', 'zeta_2'],
        'include': True
    },
    'iv-3': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'struct_map': struct_map,
        'struct_calc': struct_calc,
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l']
        },
        'category': '3r',
        'include': False
    },
    'iv-deep-3': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l'],#, 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l'],#, 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l']#, 'wg3wg2_k', 'wg3wg2_l']
        },
        'category': '3r',
        'include': False
    },
    'iv-deep-3': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l', 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l', 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l', 'wg3wg2_k', 'wg3wg2_l']
        },
        'category': '3r',
        'include': False
    },

    # NFE
    'iv-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'lnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l'
            ]
        },
        'fe_type': 'g',
        'category': 'nfe'
    },
    'iv-deep-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'lnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l',
                'w4g1_k', 'w4g1_l'
            ]
        },
        'fe_type': 'g',
        'category': 'nfe'
    },
    'iv-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'gnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l'
            ]
        },
        'fe_type': 'g',
        'category': 'nfe'
    },
    'iv-deep-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'gnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l',
                'w4g1_k', 'w4g1_l'
            ]
        },
        'fe_type': 'g',
        'category': 'nfe'
    }
}

category = 'basic'
categories = dict([(entry.get('category', None), True) for entry in models.values()]).keys()
c_ind = chr(97 + list(categories).index(category))
base_out_name = ("results_1", "lmm")
out_name = f"{base_out_name[0]}{c_ind}_{base_out_name[1]}_{category}_str"

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args: list[tuple[ModelSpec, str, str]] = []
for m_key, mod_obj in models.items():
    if 'struct_map' not in mod_obj:
        mod_obj['struct_map'] = struct_map if 'lag' not in m_key else {}
    if 'struct_calc' not in mod_obj:
        mod_obj['struct_calc'] = struct_calc if 'lag' not in m_key else []
    mod = ModelSpec(**mod_obj)
    if mod.include == False:
        continue
    if category is not None and mod.category != category:
        continue
    worker_args.append((
        mod,
        mod.panel_name if mod.panel_name is not None else def_panel_name,
        m_key
    ))

# Refactor this so we open a blank new file,
# and then with each res, mod, model_name, we append to the file instead of writing all at once at the end.
# This way, if a model fails or I interrupt, we still have the results of the previous models saved.
with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
    for args in worker_args:
        mod, _, model_name = args
        res, beta, effects_dict = run_panel(args)
        if res is None:
            print(f"Model '{model_name}' failed. Skipping.")
            continue
        output_str = format_str((res, mod, model_name))     # type: ignore
        f.write(output_str + "\n" + "=" * 80 + "\n" + "=" * 80 + "\n")
        run_res_series.append((res, mod, model_name))       # type: ignore

model_count = len(run_res_series)
if model_count:
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}. Saved to {out_name}.txt")

Running model 'iv4' as linearmodels panel IV, formula: `y` ~ `k` + `l` + `wg1_k` + `wg1_l` + year_2018 + year_2019 + year_2016 + year_2017 + year_2012 + year_2008 + year_2022 + year_2007 + year_2020 + year_2015 + year_2014 + year_2013 + year_2011 + year_2009 + year_2010 + year_2023 + year_2024 + year_2021 + [`wg1_y` ~ `w2g1_k` + `w2g1_l` + `w3g1_k` + `w3g1_l` + `w4g1_k` + `w4g1_l`]
J-statistic (rej if overidentified): 5.51, p-value: 0.357
beta_1: 0.2337SE: 0.0079,t-stat: 29.7501,p-value: 0.0000,Lower CI: 0.2183,Upper CI: 0.2491
beta_2: 0.5553SE: 0.0074,t-stat: 74.5524,p-value: 0.0000,Lower CI: 0.5407,Upper CI: 0.5699
delta_1: -0.4144SE: 0.1772,t-stat: -2.3388,p-value: 0.0193,Lower CI: -0.7617,Upper CI: -0.0671
xi_1: 0.0309SE: 0.0140, t-stat: 2.2144, p-value: 0.0268, Lower CI: 0.0036, Upper CI: 0.0583
zeta_1: 0.0371SE: 0.0135, t-stat: 2.7491, p-value: 0.0060, Lower CI: 0.0107, Upper CI: 0.0636
✅ Structural parameters extracted: beta_1, beta_2, delta_1, xi_1, zeta_1
✅ Model 'iv4' estimat

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [ ]:
from linearmodels.panel.results import PanelEffectsResults
from f_1a_run_panel import run_panel, ModelSpec, format_str

def_panel_name = "working_yearly_n"

struct_map = {
    'k': 'beta_1',
    'l': 'beta_2',
    'wd1_y': 'delta_0',
    'wd2_y': 'delta_0',
    'wd3_y': 'delta_0'
}
struct_calc = ['xi_0', 'zeta_0']
m_models = {
    # Basic models
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'basic'
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'basic'
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'basic'
    },

    # Models excluding firms which share a location
    'dd1-noz': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'noz'
    },
    'dd2-noz': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'noz'
    },
    'dd3-noz': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'noz'
    },
    'dd1-deep3-noz': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'noz'
    },
    'dd1-deep4-noz': { #+TODO: This one looks promising!
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'noz'
    },


    # Models with more instruments (high-order lags)
    'lag1-deep3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'basic'
    },
    'dd1-deep4': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l']
        },
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'basic'
    },

    # OLS 1st-stage tests
    'lag2': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'ols'
    },
    'lag3': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'ols'
    },
    'lag4': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_n',
        'category': 'ols'
    },

    # Models excluding firms which share a location
    'lag2-noz': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'ols'
    },
    'lag3-noz': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'ols'
    },
    'lag4-noz': {
        'Y': 'wd1_y',
        'X': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l'],
        'fe': ['i', 't'],
        'panel_name': 'working_yearly_no',
        'category': 'ols'
    },

    # Network fixed effects
    'dd1-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'lnfe'],
        'description': 'Distance 1 model',
        'panel_name': 'working_yearly_n',
        'category': 'nfe'
    },
    'dd1-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_n',
        'category': 'nfe'
    },
    'dd1-noz-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'lnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-noz-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-3deep-noz-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['t', 'lnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-4deep-noz-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    }
}

category = 'ols'
models = m_models
categories = dict([(entry.get('category', None), True) for entry in m_models.values()]).keys()
c_ind = chr(97 + list(categories).index(category))
base_out_name = ("results_2", "dd")
out_name = f"{base_out_name[0]}{c_ind}_{base_out_name[1]}_{category}_str"

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args: list[tuple[ModelSpec, str, str]] = []
for m_key, mod_obj in models.items():
    if category not in ['ols']:
        mod_obj['struct_map'] = struct_map
        mod_obj['struct_calc'] = struct_calc
    mod = ModelSpec(**mod_obj)
    if (not mod.include) or (category is not None and mod.category != category):
        continue
    worker_args.append((
        mod,
        mod.panel_name if mod.panel_name is not None else def_panel_name,
        m_key
    ))

# Refactor this so we open a blank new file,
# and then with each res, mod, model_name, we append to the file instead of writing all at once at the end.
# This way, if a model fails or I interrupt, we still have the results of the previous models saved.
with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
    for args in worker_args:
        mod, _, model_name = args
        res, beta, effects_dict = run_panel(args)
        if res is None:
            print(f"Model '{model_name}' failed. Skipping.")
            continue
        output_str = format_str((res, mod, model_name))     # type: ignore
        f.write(output_str + "\n" + "=" * 80 + "\n" + "=" * 80 + "\n")
        run_res_series.append((res, mod, model_name))       # type: ignore

model_count = len(run_res_series)
if model_count:
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}. Saved to {out_name}.txt")

Running model 'dd1-1s-noz' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s-noz' estimated: w2d1_k=0.312, w2d1_l=0.361
Running model 'dd1-1s-noz3' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s-noz3' estimated: w2d1_k=-0.016, w2d1_l=-0.064, w3d1_k=0.440, w3d1_l=0.707
Running model 'dd1-1s-noz4' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s-noz4' estimated: w2d1_k=0.185, w2d1_l=0.046, w3d1_k=0.470, w3d1_l=0.722, w4d1_k=-0.239, w4d1_l=-0.132
Running model 'dd1-1s' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s' estimated: w2d1_k=0.277, w2d1_l=0.397
Running model 'dd1-1s-3' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s-3' estimated: w2d1_k=0.024, w2d1_l=0.025, w3d1_k=0.373, w3d1_l=0.640
Running model 'dd1-1s-4' as panel OLS
✅ Structural parameters extracted: 
✅ Model 'dd1-1s-4' estimated: w2d1_k=0.355, w2d1_l=0.144, w3d1_k=0.392, w3d1_l=0.646, w4d1_k=-0.361, w4d1_l=-0.124
Panel regressions complete. 